# EA1 - Carga y transformaciones del dataset Iris

**Asignatura:** Proyecto Integrado III - Analitica de Datos (PREICA2602B020038)

Flujo: cargar `scikit-learn/iris` desde Hugging Face con `HFDatasetLoader`, exportar el crudo, aplicar cinco transformaciones y generar el resumen con `DatasetSummarizer`.

## 1. Importar las clases desde `../src`

Se resuelve la ruta con `pathlib` + `sys.path`. El bloque alternativo (comentado) permite ejecutar el notebook en Google Colab clonando el repositorio.

In [1]:
import sys
import pathlib

# Ruta local: la carpeta src esta un nivel arriba de notebooks.
SRC = (pathlib.Path.cwd().parent / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# --- Alternativa para Google Colab (descomentar si se ejecuta en Colab) ---
# !git clone https://github.com/Jeremy-Pedraza/PREICA2602B020038-Analisis-de-Datos.git
# SRC = pathlib.Path('/content/PREICA2602B020038-Analisis-de-Datos/EA1/src').resolve()
# sys.path.insert(0, str(SRC))

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

print('Clases importadas desde:', SRC)

Clases importadas desde: G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\EA1\src


## 2. Cargar `scikit-learn/iris` a un DataFrame

In [2]:
loader = HFDatasetLoader(dataset_id='scikit-learn/iris', split='train')
df_crudo = loader.to_dataframe()
print('Metadatos:', loader.get_metadata())
df_crudo.head()

README.md:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

C:\laragon\bin\python\python-3.10\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Desarrollador\.cache\huggingface\hub\datasets--scikit-learn--iris. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Iris.csv:   0%|          | 0.00/5.11k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Metadatos: {'dataset_id': 'scikit-learn/iris', 'split': 'train', 'config': None, 'revision': None, 'fecha_consulta': '2026-10-24T16:00:42.397393+00:00', 'n_filas': 150, 'n_columnas': 6, 'es_muestra': False}


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 3. Exportar el crudo a `resultados/iris_crudo.csv`

In [3]:
RESULTADOS = (pathlib.Path.cwd().parent / 'resultados')
RESULTADOS.mkdir(parents=True, exist_ok=True)
ruta_crudo = RESULTADOS / 'iris_crudo.csv'
df_crudo.to_csv(ruta_crudo, index=False, encoding='utf-8')
print('Crudo exportado a:', ruta_crudo)

Crudo exportado a: G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\EA1\resultados\iris_crudo.csv


## 4. Descripcion basica

In [4]:
print('shape:', df_crudo.shape)
print('\ncolumns:', list(df_crudo.columns))
print('\ndtypes:')
print(df_crudo.dtypes)
print('\nnulos por columna:')
print(df_crudo.isna().sum())
df_crudo.head()

shape: (150, 6)

columns: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']

dtypes:
Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species           object
dtype: object

nulos por columna:
Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 5. Transformaciones

Cinco transformaciones, cada una en su celda, asignando a un DataFrame nuevo que se encadena al siguiente paso.

### T1 - Renombrar columnas a snake_case en espanol

In [5]:
mapa_columnas = {
    'SepalLengthCm': 'largo_sepalo', 'SepalWidthCm': 'ancho_sepalo',
    'PetalLengthCm': 'largo_petalo', 'PetalWidthCm': 'ancho_petalo',
    'Species': 'especie',
    # Nombres alternativos segun la version del dataset:
    'sepal_length': 'largo_sepalo', 'sepal_width': 'ancho_sepalo',
    'petal_length': 'largo_petalo', 'petal_width': 'ancho_petalo',
    'species': 'especie', 'target': 'especie',
}
df_t1 = df_crudo.rename(columns=mapa_columnas)
# Eliminar columna de indice si viene incluida.
df_t1 = df_t1.drop(columns=[c for c in ['Id', 'id', 'Unnamed: 0'] if c in df_t1.columns])
print('Columnas tras T1:', list(df_t1.columns))
df_t1.head()

Columnas tras T1: ['largo_sepalo', 'ancho_sepalo', 'largo_petalo', 'ancho_petalo', 'especie']


,largo_sepalo,ancho_sepalo,largo_petalo,ancho_petalo,especie
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### T2 - Limpieza: eliminar duplicados y nulos

In [6]:
filas_inicio = len(df_t1)
df_t2 = df_t1.drop_duplicates().dropna().reset_index(drop=True)
print(f'Duplicados/nulos eliminados: {filas_inicio - len(df_t2)}')
print(f'Filas: {filas_inicio} -> {len(df_t2)}')
df_t2.head()

Duplicados/nulos eliminados: 3
Filas: 150 -> 147


,largo_sepalo,ancho_sepalo,largo_petalo,ancho_petalo,especie
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### T3 - Normalizacion de texto en `especie`

In [7]:
df_t3 = df_t2.copy()
df_t3['especie'] = (
    df_t3['especie'].astype(str)
    .str.replace('Iris-', '', regex=False)
    .str.strip()
    .str.capitalize()
)
print('Especies normalizadas:', df_t3['especie'].unique())
df_t3.head()

Especies normalizadas: ['Setosa' 'Versicolor' 'Virginica']


,largo_sepalo,ancho_sepalo,largo_petalo,ancho_petalo,especie
0,5.1,3.5,1.4,0.2,Setosa
1,4.9,3.0,1.4,0.2,Setosa
2,4.7,3.2,1.3,0.2,Setosa
3,4.6,3.1,1.5,0.2,Setosa
4,5.0,3.6,1.4,0.2,Setosa


### T4 - Columnas calculadas: areas y razon

In [8]:
df_t4 = df_t3.copy()
df_t4['area_sepalo'] = df_t4['largo_sepalo'] * df_t4['ancho_sepalo']
df_t4['area_petalo'] = df_t4['largo_petalo'] * df_t4['ancho_petalo']
df_t4['razon_petalo_sepalo'] = df_t4['area_petalo'] / df_t4['area_sepalo']
df_t4.head()

,largo_sepalo,ancho_sepalo,largo_petalo,ancho_petalo,especie,area_sepalo,area_petalo,razon_petalo_sepalo
0,5.1,3.5,1.4,0.2,Setosa,17.85,0.28,0.015686
1,4.9,3.0,1.4,0.2,Setosa,14.70,0.28,0.019048
2,4.7,3.2,1.3,0.2,Setosa,15.04,0.26,0.017287
3,4.6,3.1,1.5,0.2,Setosa,14.26,0.30,0.021038
4,5.0,3.6,1.4,0.2,Setosa,18.00,0.28,0.015556


### T5 - Categorizacion de `tamano_flor` con `pd.cut`

In [9]:
import pandas as pd

df_t5 = df_t4.copy()
df_t5['tamano_flor'] = pd.cut(
    df_t5['area_petalo'], bins=3, labels=['Pequena', 'Mediana', 'Grande']
)
print(df_t5['tamano_flor'].value_counts())
df_transformado = df_t5
df_transformado.head()

tamano_flor
Pequena    68
Mediana    51
Grande     28
Name: count, dtype: int64


,largo_sepalo,ancho_sepalo,largo_petalo,ancho_petalo,especie,area_sepalo,area_petalo,razon_petalo_sepalo,tamano_flor
0,5.1,3.5,1.4,0.2,Setosa,17.85,0.28,0.015686,Pequena
1,4.9,3.0,1.4,0.2,Setosa,14.70,0.28,0.019048,Pequena
2,4.7,3.2,1.3,0.2,Setosa,15.04,0.26,0.017287,Pequena
3,4.6,3.1,1.5,0.2,Setosa,14.26,0.30,0.021038,Pequena
4,5.0,3.6,1.4,0.2,Setosa,18.00,0.28,0.015556,Pequena


## 6. Exportar el transformado a `resultados/iris_transformado.csv`

In [10]:
ruta_transformado = RESULTADOS / 'iris_transformado.csv'
df_transformado.to_csv(ruta_transformado, index=False, encoding='utf-8')
print('Transformado exportado a:', ruta_transformado)
print('shape final:', df_transformado.shape)

Transformado exportado a: G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\EA1\resultados\iris_transformado.csv
shape final: (147, 9)


## 7. Resumen con `DatasetSummarizer` (sobre el transformado)

In [11]:
summarizer = DatasetSummarizer(df_transformado, metadata=loader.get_metadata())
ruta_resumen = summarizer.save(RESULTADOS / 'resumen_dataset.txt')
print('Resumen guardado en:', ruta_resumen)

Resumen guardado en:

 G:\Mi unidad\iUDigital Software\2026 S2\ProgramacionAnalisisdeDatos\EV1\EA1\resultados\resumen_dataset.txt


## 8. Verificacion: leer e imprimir el `.txt` generado

In [12]:
print(ruta_resumen.read_text(encoding='utf-8'))

RESUMEN DEL DATASET
1. IDENTIFICACION
------------------------------------------------------------
   dataset_id      : scikit-learn/iris
   split           : train
   configuracion   : None
   revision        : None
   fecha_consulta  : 2026-10-24T16:00:42.397393+00:00
   es_muestra      : False

2. DIMENSIONES
------------------------------------------------------------
   filas    : 147
   columnas : 9

3. COLUMNAS Y TIPOS
------------------------------------------------------------
   largo_sepalo                   float64
   ancho_sepalo                   float64
   largo_petalo                   float64
   ancho_petalo                   float64
   especie                        object
   area_sepalo                    float64
   area_petalo                    float64
   razon_petalo_sepalo            float64
   tamano_flor                    category

4. VALORES NULOS POR COLUMNA
------------------------------------------------------------
   largo_sepalo                         